# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
        ... # To-Do: fill in metadata and page_content appropriately
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])


## 2) Split into chunks


In [ ]:
chunk_size = ... # To-Do: specify chunk size, a way to estimate is to divide average document length by desired number of chunks
chunk_overlap = # To-Do: specify chunk overlap, a way to estimate is to take a small percentage of chunk size

splitter = RecursiveCharacterTextSplitter(
    ... # To-Do: fill in chunk_size and chunk_overlap appropriately
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])


## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    ... #To-Do: fill in splits and embeddings appropriately, you have to use DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

Retriever ready


## 4) Build the RAG chain


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA  # on latest stack

llm_id = "google/flan-t5-small"
hf = pipeline(
    ... # To-Do: fill in the pipeline appropriately, remember that you are using a text2text-generation task
)

llm = HuggingFacePipeline(pipeline=hf)

qa = RetrievalQA.from_chain_type(
    ... # To-Do: fill in the llm and retriever with the following parameters: llm, retriever, chain_type
)

print("RAG chain ready")


## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa(...) #To-Do: fill in the question appropriately

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))